# RQ2 — Anteprima V1: selezione bilanciata fra cloudbed

Eseguire **dopo** i sette test sintetici PyCharm `test_rq2_partition_preview_v1.py`. La V0 è preservata. Questa versione corregge la concentrazione di tutte le discovery su cloudbed-2 e il rischio di finestre consecutive: esclude come discovery i singleton che toccano altre finestre, preserva un test per classe su ogni cloudbed dove essa compare, minimizza lo sbilanciamento discovery e poi la prossimità dei casi fra fasi. Tie-break con SHA fisso, senza consultare predizioni.

**Lettura GT controllata nell'audit offline, nessuna etichetta per caso mostrata, nessuna telemetria, nessuna run Gemini/Claude, nessuna scrittura.** Status `DRAFT_NOT_EXECUTABLE`. Se sul Market reale i vincoli risultano impossibili, il notebook fallirà e non inventerà la partizione.


In [ ]:
# FASE 2 RQ2 — preview V1 bilanciato, non executable e senza API.
# L'algoritmo usa GT solo per stratificazione in modulo valutativo isolato;
# l'output contiene ID+fase e conteggi aggregati, mai ID->classe corretta.
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
DATA = Path("/content/drive/MyDrive/Tesi_OpenRCA")
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
for relative in (
    "Market.zip",
    "data/processed/split_market_v1.csv",
    "data/processed/benchmark_inputs_market.csv",
):
    if not (DATA / relative).is_file():
        raise FileNotFoundError("Input congelato mancante: " + relative)

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Clone OOD non-Git: non sovrascrivo")
    branch = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if branch != BRANCH:
        raise RuntimeError("Branch inatteso: non cambio automaticamente")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("Modifiche locali nel clone: non eseguo pull")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True, text=True, capture_output=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         BRANCH, URL, str(REPO)],
        check=True, text=True, capture_output=True
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("=== RQ2 — ANTEPRIMA V1, BILANCIAMENTO CLOUDBED ===")
print("Branch:", BRANCH)
print("Commit:", commit)

env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
res = subprocess.run(
    [sys.executable, "-m", "evaluation.rq2_partition_preview_v1",
     "--data-dir", str(DATA)],
    cwd=str(REPO), env=env, text=True, capture_output=True
)
if res.returncode:
    raise RuntimeError("V1 non fattibile/verificato su Market reale:\n" + res.stderr[-5000:])
try:
    report = json.loads(res.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("Manca JSON verificabile della preview V1") from exc

if not (
    report.get("audit") == "rq2_partition_preview_balanced_v1"
    and report.get("status") == "DRAFT_NOT_EXECUTABLE"
    and report.get("source_verified") is True
    and report.get("discovery_query_count") == 4
    and report.get("test_query_count") == 11
    and report.get("discovery_window_groups") == 4
    and report.get("test_window_groups") == 10
    and report.get("model_api_calls") == 0
    and report.get("files_written") is False
    and report.get("read_ground_truth") is True
    and report.get("read_telemetry") is False
    and report.get("emits_case_label_mapping") is False
):
    raise RuntimeError("Indicatori inattesi nel risultato V1")
if not (
    len(report["discovery_case_ids"]) == len(set(report["discovery_case_ids"]))
    and len(report["test_case_ids"]) == len(set(report["test_case_ids"]))
    and not (set(report["discovery_case_ids"]) & set(report["test_case_ids"]))
    and len(set(report["discovery_case_ids"]) | set(report["test_case_ids"])) == 15
):
    raise RuntimeError("Duplicazione, overlap discovery/test o casi mancanti")
if any(set(g) != {"phase","case_ids"} for g in report["window_groups"]):
    raise RuntimeError("Output gruppo con campi non autorizzati")
if not all(
    x["discovery_queries"] == 1 for x in report["class_balance_aggregate_only"].values()
):
    raise RuntimeError("Non esiste esattamente una discovery per classe")
print("\n=== RISULTATO V1 — SOLO PROPOSTA, NON PIANO CONGELATO ===")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nZERO API, nessun piano salvato, nessun mapping di etichette per caso.")


## Dopo l'esecuzione

Riportare il JSON completo, specialmente `discovery_by_cloudbed`, `cross_phase_near_pairs_3600s`, `min_discovery_test_gap_same_cloudbed_seconds`, `discovery_case_ids`, `test_case_ids`. Non congelare alcun piano fino a revisione congiunta. I proxy temporali non dimostrano indipendenza fisica.
